First the relevant libraries are imported into the notebook

In [6]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem import PandasTools

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

from data_profiling import ProfileReport   

Next the data set is loaded and checked to ensure it was imported properly

In [ ]:
from urllib.request import urlretrieve

url = "https://github.com/Natashasia/CHE1147-Group-6/blob/main/tox21_10k_data_all.sdf"

sdf_url = url.replace("github.com/", "raw.githubusercontent.com/").replace("/blob/", "/")
sdf_path = "tox21_10k_data_all.sdf"
urlretrieve(sdf_url, sdf_path)

df = PandasTools.LoadSDF(
    sdf_path,
    smilesName="smiles",
    molColName="mol",
    includeFingerprints=True
)

df.head()

OSError: [Errno 22] Invalid argument: 'https://github.com/Natashasia/CHE1147-Group-6/blob/main/tox21_10k_data_all.sdf'

The dataset is checked for duplicate molecules

In [ ]:
unique_smiles = df['smiles'].nunique()
num_smiles = df['smiles'].shape[0]
print(f'Unique SMILES: {unique_smiles}')
print(f'Total SMILES: {num_smiles}')

No duplicates were found. Now a sub data frame can be made from the dataa set only including the areas of interest

In [ ]:
df_of_study = df[['SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id','smiles']]
df_of_study.head()

In [ ]:
#Declare a function for canonicalizing isomeric SMILES
def con_smiles(smile, isomeric:bool=True):
    mol = Chem.MolFromSmiles(smile)
    canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=isomeric)
    return canonical_smiles

The smile format of molecules are added to the dataframe in a new column

In [ ]:
PandasTools.AddMoleculeColumnToFrame(df, smilesCol='smiles', molCol = 'molecule')
df.head()

A function is defined to apply key molecular descriptors to the dataset, it is then run to find and assign these values to the respective molecules

In [ ]:
def calculate_key_descriptors(mol):
    descriptors = {
        'MolWt': Descriptors.MolWt(mol),
        'LogP' : Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'RotBonds' : Descriptors.NumRotatableBonds(mol)
    }
    return descriptors

descriptor_data = []
for row, mol in df['molecule'].items():
    if mol is None:
        desc = {
            'MolWt': np.nan,
            'LogP': np.nan,
            'TPSA': np.nan,
            'NumHDonors': np.nan,
            'NumHAcceptors': np.nan,
            'RotBonds': np.nan
        }
    else:
        desc = calculate_key_descriptors(mol)
    descriptor_data.append(desc)

descriptor_df = pd.DataFrame(descriptor_data, index=df.index)

df_with_descriptors = pd.concat([df, descriptor_df], axis=1)

df_with_descriptors.head()

Information about the dataset with descriptors is checked

In [ ]:
df_with_descriptors.info()

In [ ]:
df_with_descriptors.describe()

Next the key features for comparison are plotted

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribution of Key Molecular Descriptors', fontsize=16, fontweight='bold')

descriptors_to_plot = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'RotBonds']

for i, descriptor in enumerate(descriptors_to_plot):
    row = i // 3
    col = i % 3

    axes[row, col].hist(df_with_descriptors[descriptor].dropna(), bins=30, color='skyblue', edgecolor='black')

    mean_val = df_with_descriptors[descriptor].mean()
    median_val = df_with_descriptors[descriptor].median()
    axes[row, col].axvline(mean_val, color='red', linestyle='dashed', linewidth=1, label=f'Mean: {mean_val:.2f}')
    axes[row, col].axvline(median_val, color='green', linestyle='dashed', linewidth=1, label=f'Median: {median_val:.2f}')
    axes[row, col].set_title(f'{descriptor} Distribution', fontsize=14)
    axes[row, col].set_xlabel(descriptor, fontsize=12)
    axes[row, col].set_ylabel('Density', fontsize=12)
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Next the correlations between the key descriptors are evaluated

In [ ]:
descriptor_columns = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'RotBonds']
correlation_matrix = df_with_descriptors[descriptor_columns].corr()

correlation_matrix

And plotted in a heatmap for convenient visual representation

In [ ]:
plt.figure(figsize=(10, 8))

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True,cmap='coolwarm', center=0, square=True,fmt='.3f',cbar_kws={'label': 'Correlation Coefficient'})

plt.title('Molecular Descriptor Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

Using the ProfileReport feature from data_profilling (previously ydata-profiling) a complete EDA report is generated and then displayed inline with the Jupyter Notebook

In [ ]:
profile = ProfileReport(df_with_descriptors, title="Tox21 Dataset EDA Report", explorative=True)
profile

Finally the EDA report for the Tox 21 data set is exported into an html file for easy viewing and sharing

In [ ]:
profile.to_file("tox21_eda_report.html")